In [1]:
import qlib
import pandas as pd

from qlib.data.dataset.handler import DataHandlerLP

qlib.init(provider_uri="~/.qlib/qlib_data/cn_data")


[61028:MainThread](2026-02-06 20:30:09,464) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[61028:MainThread](2026-02-06 20:30:09,948) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[61028:MainThread](2026-02-06 20:30:09,949) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/chainsmoker/.qlib/qlib_data/cn_data')}


In [2]:
from qlib.data import D

instruments = D.instruments(market='all')
start_time="2025-01-01"
end_time="2025-12-31"
fit_start_time="2025-01-01"
fit_end_time="2025-8-31"
test_start_time = "2025-09-01"
test_end_time = "2025-10-31"

stock_list = D.list_instruments(instruments=instruments,
                                start_time=start_time,
                                end_time=end_time,
                                as_list=True)
print(stock_list[-5:])


['SZ301667', 'SZ301668', 'SZ301678', 'SZ301687', 'SZ302132']


In [3]:
from qlib.data.dataset.loader import QlibDataLoader

data_loader = QlibDataLoader(
    config={
        "feature": [
            "$open",
            "$high",
            "$low",
            "$close",
            "$volume",
        ],
        "label": [
            "Ref($close, -2) / Ref($close, -1) - 1",  # 未来1日收益（示例）
        ],
    }
)


In [4]:
handler = DataHandlerLP(
    instruments=instruments,
    start_time=start_time,
    end_time=end_time,
    data_loader=data_loader,
)


[61028:MainThread](2026-02-06 20:30:20,196) INFO - qlib.timer - [log.py:127] - Time cost: 3.640s | Loading data Done
[61028:MainThread](2026-02-06 20:30:20,197) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done
[61028:MainThread](2026-02-06 20:30:20,198) INFO - qlib.timer - [log.py:127] - Time cost: 3.642s | Init data Done


In [6]:
# 取出“日级特征表”和“日级标签表”
df_x = handler.fetch(col_set="feature")
df_y = handler.fetch(col_set="label")

print("=== features（日级 MultiIndex 表）===")
print(df_x.head())
print("index:", df_x.index.names, "shape:", df_x.shape, "columns:", list(df_x.columns))

print("=== label（日级 MultiIndex 表）===")
print(df_y.head())
print("index:", df_y.index.names, "shape:", df_y.shape, "columns:", list(df_y.columns))


=== features（日级 MultiIndex 表）===
                           $open      $high       $low     $close    $volume
datetime   instrument                                                       
2025-01-02 BJ920000    18.299999  18.500000  17.379999  17.900000   993934.0
           BJ920001    16.350000  17.440001  16.000000  16.600000  3958586.0
           BJ920002    67.500000  69.989998  66.989998  67.980003   791137.0
           BJ920006    24.000000  24.980000  23.510000  24.820000  5070593.0
           BJ920008    30.280001  30.660000  28.969999  29.280001  1103154.0
index: ['datetime', 'instrument'] shape: (1273246, 5) columns: ['$open', '$high', '$low', '$close', '$volume']
=== label（日级 MultiIndex 表）===
                       Ref($close, -2) / Ref($close, -1) - 1
datetime   instrument                                       
2025-01-02 BJ920000                                -0.016234
           BJ920001                                -0.128030
           BJ920002                        

In [7]:
import numpy as np
from qlib.data.dataset import TSDatasetH

L = 60
dataset = TSDatasetH(
    handler=handler,
    segments={
        "train": (fit_start_time, fit_end_time),
        "test":  (test_start_time, test_end_time),
    },
    step_len=L,
)

# 准备 train 段样本（返回一个可索引对象）
train_set = dataset.prepare("train")
test_set = dataset.prepare("test")

print("train_set length:", len(train_set))
print("test_set length:", len(test_set))


train_set length: 841361
test_set length: 204956


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class QlibOHLCVDataset(Dataset):
    """
    输入: qlib_ts_set[i] -> numpy (L, 6) = [O,H,L,C,V,label]
    输出：
      x: torch.FloatTensor (L, 5)
      y: torch.FloatTensor ()  (标量)
    """
    def __init__(self, qlib_ts_set, valid_idx, ohlcv_dim=5, label_col=5):
        self.ds = qlib_ts_set
        self.valid_idx = valid_idx # 只提取不为Nan的数据
        self.ohlcv_dim = ohlcv_dim
        self.label_col = label_col

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, j):
        i = int(self.valid_idx[j])
        s = self.ds[i]  # numpy (L,6)
        x = torch.from_numpy(s[:, :self.ohlcv_dim].astype(np.float32))   # (L,5)
        y = torch.tensor(np.float32(s[-1, self.label_col]), dtype=torch.float32)  # scalar
        return x, y


In [9]:
def build_valid_indices(qlib_ts_set, ohlcv_dim=5, label_col=5):
    valid = []
    for i in range(len(qlib_ts_set)):
        s = qlib_ts_set[i]  # (L,6)
        x = s[:, :ohlcv_dim]
        y = s[-1, label_col]
        if np.isfinite(x).all() and np.isfinite(y):
            valid.append(i)
    return np.array(valid, dtype=np.int64)


In [10]:
train_set = dataset.prepare("train")
test_set  = dataset.prepare("test")

valid_train = build_valid_indices(train_set)
valid_test  = build_valid_indices(test_set)

print("train kept:", len(valid_train), " / ", len(train_set), " ratio:", len(valid_train)/len(train_set))
print("test kept:",  len(valid_test),  " / ", len(test_set),  " ratio:", len(valid_test)/len(test_set))

torch_train = QlibOHLCVDataset(train_set, valid_train)
torch_test  = QlibOHLCVDataset(test_set,  valid_test)


train kept: 520884  /  841361  ratio: 0.6190969155927123
test kept: 201104  /  204956  ratio: 0.9812057222037901


In [11]:
train_loader = DataLoader(torch_train, batch_size=64, shuffle=True, drop_last=True)
test_loader  = DataLoader(torch_test,  batch_size=64, shuffle=False)


In [12]:
import torch.nn as nn

from alpha158_templates import build_templates
from alpha158_ops import eval_graph
from alpha158_rolling import align_to_length

class TemplateFactorExtractor(nn.Module):
    """Use 41 template graphs to compute 41 factors."""
    def __init__(self):
        super().__init__()
        templates = build_templates()
        self.items = []
        for t in templates:
            rep = t["names"][0]
            params = t.get("name_params", {}).get(rep, {})
            self.items.append((t["template_id"], t["graph"], params))
        self.feat_dim = len(self.items)

    def forward(self, x_raw):
        # x_raw: (B, L, 5)
        open_ = x_raw[..., 0]
        high_ = x_raw[..., 1]
        low_ = x_raw[..., 2]
        close_ = x_raw[..., 3]
        volume_ = x_raw[..., 4]
        # vwap not provided in loader; approximate with typical price
        vwap_ = (open_ + high_ + low_ + close_) / 4.0

        variables = {
            "open_": open_,
            "high_": high_,
            "low_": low_,
            "close_": close_,
            "volume_": volume_,
            "vwap_": vwap_,
        }

        feats = []
        lengths = []
        for _, graph, params in self.items:
            out = eval_graph(graph, variables, params)
            feats.append(out)
            lengths.append(out.size(1))

        Lp = min(lengths)
        feats = [align_to_length(f, Lp, dim=1) for f in feats]
        return torch.stack(feats, dim=-1)


In [13]:
class LSTMPredictor(nn.Module):
    def __init__(self, feat_dim, hidden=64, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_size=feat_dim, hidden_size=hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, feats):
        out, _ = self.lstm(feats)         # (B, Lp, hidden)
        last = out[:, -1, :]              # (B, hidden)
        last = self.drop(last)
        yhat = self.fc(last).squeeze(-1)  # (B,)
        return yhat


class End2EndModel(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.extractor = TemplateFactorExtractor()
        self.predictor = LSTMPredictor(feat_dim=self.extractor.feat_dim, hidden=hidden)

    def forward(self, x_raw):
        feats = self.extractor(x_raw)
        return self.predictor(feats)


In [16]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model = End2EndModel(hidden=64).to(device)

xb, yb = next(iter(train_loader))        # xb: (B, 60, 5), yb: (B,)
xb = xb.to(device).detach().requires_grad_(True)
yb = yb.to(device)

# Factor-level gradient check
feats = model.extractor(xb)
feats.retain_grad()
yhat = model.predictor(feats)
loss = F.mse_loss(yhat, yb)
loss.backward()

print("xb:", xb.shape, "yb:", yb.shape)
print("feats:", feats.shape)
print("loss:", float(loss))
print("xb.grad is None?", xb.grad is None)
print("grad mean abs:", float(xb.grad.abs().mean()))
print("grad max abs:", float(xb.grad.abs().max()))
print("grad has nan?", bool(torch.isnan(xb.grad).any()))

# per-factor gradient magnitude
if feats.grad is not None:
    per_factor = feats.grad.abs().mean(dim=(0,1))
    print("per-factor grad mean:", per_factor)
    print("zero-grad factors:", int((per_factor == 0).sum()))
else:
    print("feats.grad is None")


xb: torch.Size([64, 60, 5]) yb: torch.Size([64])
feats: torch.Size([64, 55, 41])
loss: 0.0025248818565160036
xb.grad is None? False
grad mean abs: 3.5170489809388528e-06
grad max abs: 0.0014738411409780383
grad has nan? False
per-factor grad mean: tensor([5.8350e-07, 8.5483e-07, 4.5255e-07, 2.5368e-07, 1.3705e-07, 6.4097e-07,
        2.9702e-07, 4.2440e-07, 8.1358e-07, 5.3435e-07, 5.0435e-07, 1.0028e-07,
        1.7901e-07, 2.4268e-07, 5.1316e-07, 3.1664e-07, 6.7251e-07, 1.0253e-06,
        1.0364e-07, 4.2598e-07, 3.9013e-07, 4.4057e-07, 1.3608e-07, 4.4210e-07,
        1.2162e-07, 3.4027e-07, 3.4686e-07, 5.7306e-07, 6.4813e-07, 1.1005e-06,
        4.2556e-07, 7.3655e-07, 4.8028e-07, 2.3478e-07, 4.2006e-07, 6.7164e-07,
        2.8984e-07, 2.3954e-07, 1.5793e-07, 4.8264e-07, 6.8093e-07])
zero-grad factors: 0
